# Deep Learning Classification Boilerplate

Binary and multiclass classification with Keras. Includes preprocessing, class weighting, early stopping, evaluation, and lightweight MLflow logging.

In [ ]:
# ============================================================
# 0. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    classification_report,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

import mlflow
import mlflow.tensorflow

In [ ]:
# ============================================================
# 1. BINARY DATA
# ============================================================

X, y = make_classification(
    n_samples=5000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    weights=[0.80, 0.20],
    random_state=42,
)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42,
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ============================================================
# 2. BINARY CLASS WEIGHTS
# ============================================================

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)

class_weights = dict(
    zip(classes, weights)
)

print(class_weights)

In [ ]:
# ============================================================
# 3. BINARY MODEL
# ============================================================

binary_model = keras.Sequential([
    layers.Input(
        shape=(X_train_scaled.shape[1],)
    ),
    layers.Dense(
        64,
        activation="relu"
    ),
    layers.Dropout(0.30),
    layers.Dense(
        32,
        activation="relu"
    ),
    layers.Dropout(0.20),
    layers.Dense(
        1,
        activation="sigmoid"
    ),
])

binary_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

binary_model.summary()

In [ ]:
# ============================================================
# 4. BINARY TRAINING
# ============================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

history = binary_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(
        X_val_scaled,
        y_val
    ),
    epochs=50,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1,
)

In [ ]:
# ============================================================
# 5. BINARY EVALUATION
# ============================================================

probabilities = (
    binary_model
    .predict(X_val_scaled)
    .ravel()
)

predictions = (
    probabilities >= 0.5
).astype(int)

print(
    classification_report(
        y_val,
        predictions
    )
)

print(
    "F1:",
    f1_score(
        y_val,
        predictions
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_val,
        probabilities
    )
)

print(
    "PR-AUC:",
    average_precision_score(
        y_val,
        probabilities
    )
)

In [ ]:
# ============================================================
# 6. BINARY TRAINING CURVES
# ============================================================

plt.plot(
    history.history["loss"],
    label="train_loss"
)

plt.plot(
    history.history["val_loss"],
    label="val_loss"
)

plt.title("Binary Training Loss")
plt.legend()
plt.show()

In [ ]:
# ============================================================
# 7. LIGHTWEIGHT MLFLOW - BINARY
# ============================================================

# Point this to your tracking server when required
# mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("keras-classification")

with mlflow.start_run(
    run_name="keras-binary"
):
    mlflow.log_params({
        "problem_type": "binary",
        "hidden_1": 64,
        "hidden_2": 32,
        "batch_size": 64,
    })

    mlflow.log_metrics({
        "val_f1": f1_score(
            y_val,
            predictions
        ),
        "val_roc_auc": roc_auc_score(
            y_val,
            probabilities
        ),
        "val_pr_auc": average_precision_score(
            y_val,
            probabilities
        ),
    })

    mlflow.tensorflow.log_model(
        binary_model,
        name="model"
    )

In [ ]:
# ============================================================
# 8. MULTICLASS DATA
# ============================================================

X_multi, y_multi = make_classification(
    n_samples=5000,
    n_features=20,
    n_informative=14,
    n_redundant=2,
    n_classes=4,
    n_clusters_per_class=1,
    random_state=42,
)

Xm_train, Xm_temp, ym_train, ym_temp = train_test_split(
    X_multi,
    y_multi,
    test_size=0.30,
    stratify=y_multi,
    random_state=42,
)

Xm_val, Xm_test, ym_val, ym_test = train_test_split(
    Xm_temp,
    ym_temp,
    test_size=0.50,
    stratify=ym_temp,
    random_state=42,
)

multi_scaler = StandardScaler()

Xm_train_scaled = multi_scaler.fit_transform(Xm_train)
Xm_val_scaled = multi_scaler.transform(Xm_val)
Xm_test_scaled = multi_scaler.transform(Xm_test)

NUMBER_OF_CLASSES = len(
    np.unique(ym_train)
)

print("Classes:", NUMBER_OF_CLASSES)

In [ ]:
# ============================================================
# 9. MULTICLASS MODEL
# ============================================================

multiclass_model = keras.Sequential([
    layers.Input(
        shape=(Xm_train_scaled.shape[1],)
    ),
    layers.Dense(
        64,
        activation="relu"
    ),
    layers.Dropout(0.30),
    layers.Dense(
        32,
        activation="relu"
    ),
    layers.Dense(
        NUMBER_OF_CLASSES,
        activation="softmax"
    ),
])

multiclass_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

multiclass_model.summary()

In [ ]:
# ============================================================
# 10. MULTICLASS TRAINING
# ============================================================

multi_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

multi_history = multiclass_model.fit(
    Xm_train_scaled,
    ym_train,
    validation_data=(
        Xm_val_scaled,
        ym_val
    ),
    epochs=50,
    batch_size=64,
    callbacks=[multi_early_stopping],
    verbose=1,
)

In [ ]:
# ============================================================
# 11. MULTICLASS EVALUATION
# ============================================================

multi_probabilities = (
    multiclass_model
    .predict(Xm_val_scaled)
)

multi_predictions = np.argmax(
    multi_probabilities,
    axis=1
)

print(
    classification_report(
        ym_val,
        multi_predictions
    )
)

macro_f1 = f1_score(
    ym_val,
    multi_predictions,
    average="macro"
)

print(
    "Macro F1:",
    macro_f1
)

In [ ]:
# ============================================================
# 12. LIGHTWEIGHT MLFLOW - MULTICLASS
# ============================================================

with mlflow.start_run(
    run_name="keras-multiclass"
):
    mlflow.log_params({
        "problem_type": "multiclass",
        "number_of_classes": NUMBER_OF_CLASSES,
        "hidden_1": 64,
        "hidden_2": 32,
        "batch_size": 64,
    })

    mlflow.log_metric(
        "val_macro_f1",
        macro_f1
    )

    mlflow.tensorflow.log_model(
        multiclass_model,
        name="model"
    )

In [ ]:
# ============================================================
# 13. SAVE / LOAD KERAS MODEL
# ============================================================

binary_model.save(
    "binary_model.keras"
)

loaded_binary_model = (
    keras.models.load_model(
        "binary_model.keras"
    )
)

sample_probability = (
    loaded_binary_model
    .predict(
        X_test_scaled[:1]
    )
    .ravel()[0]
)

print(
    "Sample probability:",
    float(sample_probability)
)